# Entrenamiento · detector de INYECCIÓN / JAILBREAK (Fase 4A)

- Datos: `data/injection.csv` = deepset/prompt-injections + jackhhao/jailbreak-classification + 200 ejemplos propios en español.
- Split 80/20 estratificado por etiqueta × grupo de idioma (seed 42). El **test en español** son los ejemplos propios del 20 %.
- CV 5-fold estratificada en train para elegir C (métrica: average precision).
- El **umbral** se elige con las predicciones fuera de fold del *train* (recall ≥ 0.85 con la máxima precisión); el test no se usa para elegir nada.

In [1]:
import sys, warnings, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib, sklearn
import mlutils as U
U.set_seeds()
pd.set_option("display.max_colwidth", 120)


In [2]:
train, test = U.load_injection_split()
print("train", len(train), " test", len(test))
pd.crosstab([train["group"]], train["label"]).rename(columns={0: "benigno", 1: "inyección"})


train 1720  test 431
Out[2]: 
label      benigno  inyección
group                        
de             114         72
en             713        652
es_propio       80         80
otros            2          7


label,benigno,inyección
group,,
de,114,72
en,713,652
es_propio,80,80
otros,2,7


In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=U.SEED)
X_text, y = train["text"].tolist(), train["label"].to_numpy()

def grid(make, X, Cs):
    rows = []
    for C in Cs:
        s = cross_val_score(make(C), X, y, cv=cv, scoring="average_precision", n_jobs=1)
        rows.append({"C": C, "ap_mean": s.mean(), "ap_std": s.std()})
    return pd.DataFrame(rows)

g_tfidf = grid(lambda C: U.tfidf_pipeline(C, class_weight="balanced"), X_text, [0.3, 1, 3, 10, 30])
best_tfidf = g_tfidf.loc[g_tfidf.ap_mean.idxmax()]
g_tfidf.round(4)


Out[3]: 
      C  ap_mean  ap_std
0   0.3   0.9698  0.0051
1   1.0   0.9816  0.0033
2   3.0   0.9841  0.0034
3  10.0   0.9847  0.0037
4  30.0   0.9842  0.0043


,C,ap_mean,ap_std
0,0.3,0.9698,0.0051
1,1.0,0.9816,0.0033
2,3.0,0.9841,0.0034
3,10.0,0.9847,0.0037
4,30.0,0.9842,0.0043


In [4]:
t0 = time.perf_counter()
X_emb = U.embed(X_text, cache="inj_train")
print(f"embeddings {X_emb.shape} en {time.perf_counter()-t0:.1f}s  (MiniLM trunca a {U.get_encoder().max_seq_length} tokens)")
g_emb = grid(U.embedding_lr, X_emb, [0.1, 0.3, 1, 3, 10, 30])
best_emb = g_emb.loc[g_emb.ap_mean.idxmax()]
g_emb.round(4)


embeddings (1720, 384) en 24.3s  (MiniLM trunca a 128 tokens)
Out[4]: 
      C  ap_mean  ap_std
0   0.1   0.9339  0.0078
1   0.3   0.9413  0.0049
2   1.0   0.9485  0.0019
3   3.0   0.9533  0.0032
4  10.0   0.9548  0.0056
5  30.0   0.9531  0.0087


,C,ap_mean,ap_std
0,0.1,0.9339,0.0078
1,0.3,0.9413,0.0049
2,1.0,0.9485,0.0019
3,3.0,0.9533,0.0032
4,10.0,0.9548,0.0056
5,30.0,0.9531,0.0087


## Umbral con predicciones fuera de fold (train)

In [5]:
oof_tfidf = cross_val_predict(U.tfidf_pipeline(best_tfidf.C, class_weight="balanced"), X_text, y, cv=cv, method="predict_proba")[:, 1]
oof_emb = cross_val_predict(U.embedding_lr(best_emb.C), X_emb, y, cv=cv, method="predict_proba")[:, 1]
thr_tfidf = U.pick_threshold(y, oof_tfidf)
thr_emb = U.pick_threshold(y, oof_emb)
pd.DataFrame({"tfidf": thr_tfidf, "embeddings": thr_emb}).T


Out[5]: 
           threshold precision    recall meets
tfidf       0.674541  0.984308  0.850801  True
embeddings  0.582444      0.92  0.850801  True


,threshold,precision,recall,meets
tfidf,0.674541,0.984308,0.850801,True
embeddings,0.582444,0.92,0.850801,True


In [6]:
from datetime import datetime, timezone
meta = {"seed": U.SEED, "sklearn_version": sklearn.__version__, "trained_at": datetime.now(timezone.utc).isoformat(),
        "train_data": "ml/data/injection.csv (split 80 %)", "n_train": len(train), "positive_label": 1}

tfidf = U.tfidf_pipeline(best_tfidf.C, class_weight="balanced").fit(X_text, y)
joblib.dump({**meta, "kind": "tfidf+logreg", "pipeline": tfidf, "C": float(best_tfidf.C),
             "threshold": thr_tfidf["threshold"]}, U.MODELS / "injection_tfidf.joblib")

lr = U.embedding_lr(best_emb.C).fit(X_emb, y)
joblib.dump({**meta, "kind": "embeddings+logreg", "embedding_model": U.EMB_MODEL, "normalize_embeddings": True,
             "clf": lr, "C": float(best_emb.C), "threshold": thr_emb["threshold"]}, U.MODELS / "injection_clf.joblib")

U.update_metrics("injection_train", {
    "n_train": len(train), "n_test": len(test), "cv": "StratifiedKFold(5, shuffle, seed=42)",
    "tfidf": {"C": float(best_tfidf.C), "cv_average_precision": round(float(best_tfidf.ap_mean), 4), "threshold_from_oof": thr_tfidf},
    "embeddings": {"C": float(best_emb.C), "cv_average_precision": round(float(best_emb.ap_mean), 4), "threshold_from_oof": thr_emb},
})
sorted(p.name for p in U.MODELS.glob("injection_*.joblib"))


Out[6]: ['injection_clf.joblib', 'injection_tfidf.joblib']


['injection_clf.joblib', 'injection_tfidf.joblib']